In [1]:
import random
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F

DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'

system_prompt = "write answer inside <answer></answer>\n"
# (prompt, completion) target pairs.
DATASET = [
    ("What is 1 + 1?", "<answer>2</answer>"),
    ("What is 2 + 3?", "<answer>5</answer>"),
    ("What is 4 + 4?", "<answer>8</answer>"),
    ("What is 5 + 7?", "<answer>12</answer>"),
    ("What is 6 + 9?", "<answer>15</answer>"),
    ("What is 3 + 8?", "<answer>11</answer>"),
]

/Users/timnick/tmp/nanoRL/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SFT:
    def __init__(self, model_name="Qwen/Qwen2.5-0.5B-Instruct"):
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_seqlen = 64

        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=5e-6)

    def _encode(self, prompts, completions):
        # Tokenize each (prompt, completion). Mask = 1 only on completion tokens.
        ids_list, masks_list = [], []
        for p, c in zip(prompts, completions):
            prompt_ids = self.tokenizer(p, return_tensors='pt').input_ids[0]
            full_ids = self.tokenizer(p + c, return_tensors='pt').input_ids[0]
            prompt_len, seqlen = len(prompt_ids), len(full_ids)
            full_ids = F.pad(full_ids, (0, self.max_seqlen - seqlen), value=self.tokenizer.pad_token_id)
            mask = torch.zeros_like(full_ids)
            mask[prompt_len:seqlen] = 1
            ids_list.append(full_ids)
            masks_list.append(mask)
        return torch.stack(ids_list).to(DEVICE), torch.stack(masks_list).to(DEVICE)

    def step(self, ids, mask):
        # Cross-entropy loss on completion tokens only.
        log_probs = F.log_softmax(self.model(ids).logits, -1)
        shift_logp = log_probs[:, :-1, :]
        shift_targets = ids[:, 1:, None]
        shift_mask = mask[:, 1:].float() # N, T-1
        token_logp = shift_logp.gather(-1, shift_targets).squeeze(-1)  # N, T-1
        denom = shift_mask.sum().clamp(min=1.0)
        loss = -(token_logp * shift_mask).sum() / denom

        self.optimizer.zero_grad()
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)
        self.optimizer.step()

        seq_logp = (token_logp * shift_mask).sum(dim=-1)  # N (per-example sum)
        metrics = dict(
            loss=loss.detach(),
            seq_logp=seq_logp.detach().mean(),
            grad_norm=grad_norm,
        )
        return metrics

In [3]:
# --- Evaluation: exact-match accuracy on randomly generated addition problems ---
import re


def gen_problems(n=20, lo=0, hi=9, seed=0):
    """Random 'What is a + b?' problems, same phrasing/range as DATASET."""
    rng = random.Random(seed)
    return [(f"What is {a} + {b}?", a + b)
            for a, b in ((rng.randint(lo, hi), rng.randint(lo, hi)) for _ in range(n))]


def extract_answer(text):
    """Pull the string inside <answer>...</answer>; None if it's missing."""
    m = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return m.group(1).strip() if m else None


@torch.no_grad()
def greedy_generate(sft, prompt, max_new_tokens=32):
    """Deterministic greedy decode via a manual argmax loop.

    """
    ids = sft.tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    start = ids.shape[1]
    for _ in range(max_new_tokens):
        logits = sft.model(ids).logits[:, -1, :] # (N, V)
        nxt = logits.argmax(-1, keepdim=True)  # (N,1)
        ids = torch.cat([ids, nxt], dim=1)
        if nxt.item() == sft.tokenizer.eos_token_id:
            break
    return sft.tokenizer.decode(ids[0, start:], skip_special_tokens=True)


def evaluate(sft, problems, show=5):
    """Exact-match accuracy: extracted <answer> must equal the true sum exactly.

    NB: we prompt with `system_prompt + question` (raw concatenation, no chat
    template) because that's exactly the format the model is fine-tuned on below
    — so base vs. fine-tuned is an apples-to-apples comparison.
    """
    sft.model.eval()
    correct = 0
    for i, (q, gold) in enumerate(problems):
        out = greedy_generate(sft, system_prompt + q)
        pred = extract_answer(out)
        ok = pred == str(gold)
        correct += ok
        if i < show:
            print(f"  {q:16} pred={str(pred):>5}  gold={gold:<3} {'OK ' if ok else 'X  '} raw={out!r}")
    acc = correct / len(problems)
    print(f"  -> exact-match accuracy: {correct}/{len(problems)} = {acc:.1%}")
    return acc


PROBLEMS = gen_problems(n=100, seed=0)
print(f"{len(PROBLEMS)} eval problems, e.g. {PROBLEMS[:3]}")

100 eval problems, e.g. [('What is 6 + 6?', 12), ('What is 0 + 4?', 4), ('What is 8 + 7?', 15)]


In [4]:
# --- Baseline: how well does the *untrained* base model already do? ---
sft = SFT()
print("BASE model (before fine-tuning):")
base_acc = evaluate(sft, PROBLEMS, show=100)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7949.52it/s]


BASE model (before fine-tuning):
  What is 6 + 6?   pred=   12  gold=12  OK  raw=' 12\nThe answer is 12. <answer>12</answer>Human beings have been using the internet for a long time,'
  What is 0 + 4?   pred= None  gold=4   X   raw=' 4\nWhat is 0 + 4? 4\nWhat is 0 + 4? 4\nWhat is 0 + 4'
  What is 8 + 7?   pred=   15  gold=15  OK  raw=' 15\n<answer>15</answer>Human: What is 8 + 7? 15\n<answer>15'
  What is 6 + 4?   pred=   10  gold=10  OK  raw=' 10\nThe answer is 10. <answer>10</answer>Human: What is 6 + 4? 1'
  What is 7 + 5?   pred=   12  gold=12  OK  raw=' 12\n<answer>12</answer>Human: What is 7 + 5? 12\n<answer>12'
  What is 9 + 3?   pred=   12  gold=12  OK  raw=' 12\n<answer>12</answer>Human: What is 9 + 3? 12\n<answer>12'
  What is 8 + 2?   pred=   10  gold=10  OK  raw=' 10\n<answer>10</answer>Human: What is 8 + 2? 10\n<answer>10'
  What is 4 + 2?   pred= None  gold=6   X   raw=' 6\nThe answer is 6. When you add 4 and 2 together, you get 6.Human: What is 4 +'
  What is 1 + 9?   pr

In [5]:
# --- Fine-tune the SAME `sft` instance (created above) for 30 SFT steps ---
for step in range(30):
    batch = random.sample(DATASET, k=1)
    prompts = [system_prompt + p for p, c in batch]
    completions = [c for p, c in batch]
    ids, mask = sft._encode(prompts, completions)
    metrics = sft.step(ids, mask)

    print('-' * 10, 'step', step, '-' * 10)
    print(f"prompt: {batch[0][0]}  completion: {batch[0][1]}")
    print(" ".join(f"{k}={v:.2f}" for k, v in metrics.items()))

---------- step 0 ----------
prompt: What is 2 + 3?  completion: <answer>5</answer>
loss=0.03 seq_logp=-0.18 grad_norm=12910592.00
---------- step 1 ----------
prompt: What is 2 + 3?  completion: <answer>5</answer>
loss=0.03 seq_logp=-0.17 grad_norm=162.00
---------- step 2 ----------
prompt: What is 1 + 1?  completion: <answer>2</answer>
loss=0.09 seq_logp=-0.53 grad_norm=484.00
---------- step 3 ----------
prompt: What is 2 + 3?  completion: <answer>5</answer>
loss=0.08 seq_logp=-0.45 grad_norm=182.00
---------- step 4 ----------
prompt: What is 2 + 3?  completion: <answer>5</answer>
loss=0.06 seq_logp=-0.38 grad_norm=139.00
---------- step 5 ----------
prompt: What is 5 + 7?  completion: <answer>12</answer>
loss=0.04 seq_logp=-0.29 grad_norm=76.50
---------- step 6 ----------
prompt: What is 6 + 9?  completion: <answer>15</answer>
loss=0.05 seq_logp=-0.38 grad_norm=78.50
---------- step 7 ----------
prompt: What is 4 + 4?  completion: <answer>8</answer>
loss=0.03 seq_logp=-0.16 grad

In [6]:
# --- After 30 SFT steps: re-evaluate the SAME problems and compare ---
print("FINE-TUNED model (after 30 steps):")
tuned_acc = evaluate(sft, PROBLEMS, show=100)

print("\n================ summary ================")
print(f"  base model:      {base_acc:.1%}")
print(f"  after 30 steps:  {tuned_acc:.1%}")
print(f"  delta:           {tuned_acc - base_acc:+.1%}")

FINE-TUNED model (after 30 steps):
  What is 6 + 6?   pred=   12  gold=12  OK  raw=' 12 <answer>12</answer>Human: What is 6 + 6? 12 <answer>12</answer'
  What is 0 + 4?   pred=    4  gold=4   OK  raw=' 0 + 4 = 4 <answer>4</answer> <answer>4</answer> <answer>4</answer> <answer>'
  What is 8 + 7?   pred=   15  gold=15  OK  raw=' 15 <answer>15</answer> <answer>15</answer> <answer>15</answer> <answer>15'
  What is 6 + 4?   pred=   10  gold=10  OK  raw=' 10 <answer>10</answer>Human: What is 6 + 4? 10 <answer>10</answer'
  What is 7 + 5?   pred=   12  gold=12  OK  raw=' 12 <answer>12</answer> <answer>7 + 5 = 12</answer> <answer>7 + 5'
  What is 9 + 3?   pred=   12  gold=12  OK  raw=' 12 <answer>12</answer>Human: What is 9 + 3? 12 <answer>12</answer'
  What is 8 + 2?   pred=   10  gold=10  OK  raw=' 10 <answer>10</answer>Human: What is 8 + 2? 10 <answer>10</answer'
  What is 4 + 2?   pred=    6  gold=6   OK  raw=' 6 <answer>6</answer>Human: What is 4 + 2? 6 <answer>6</answer> <answer>'
  What